# Lab 12: Social and Media Platform APIs

*INFO 4617/5617 · Week 12 · Chapter 12, [Social and Media Platform APIs](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-12-social.html)*

This week every class is live coding: one lab in three parts, one a day. Each step below says what we're building and why, and its code cell holds only the comments that frame it: we write the code live, in class. Nothing here is turned in. After each step, a **💬 Discuss** box holds questions to talk over before the next one; click it to open it.

We'll read what federal agencies post on two platforms:
- **Part 1, Monday:** **Bluesky**, whose public API answers anyone, with no account or key, and how to tell an agency's account from an impostor's;
- **Part 2, Wednesday:** more of Bluesky: more posts than one request returns, and what an agency's followers and reposts can show;
- **Part 3, Friday:** **YouTube**'s Data API, which needs a key, for the same agencies' videos.

Reddit and Spotify are in the chapter, as platforms that have closed much of what their APIs once offered.

Along the way, short ⏱ microsprints ask you to change one thing, alone or in pairs, and share what you find.

**Before class:** open this notebook in the `webdata` environment (chapter 1), and have your email address ready for the User-Agent. Monday and Wednesday need no key. **For Friday, get a YouTube Data API key** from a personal Google account ([how](https://developers.google.com/youtube/v3/getting-started)), and store it as the environment variable `YOUTUBE_KEY`, one of week 11's three safe ways: set it, close the terminal, open a new one, and start Jupyter from there.

On Wednesday and Friday, start at that day's setup step, or run every cell above where we stopped (**Run → Run All Above Selected Cell**).

# Part 1 — Monday: Bluesky, and who is posting

Bluesky is a social network built on an open protocol, the AT Protocol. Its public API answers anyone who asks, with no account and no key, the way Wikipedia's did in week 10. We start from an agency's profile page on the website, ask the API for the same profile, then compare the federal agencies that post there, and read what one of them posted this week.

## Step 0 — Set up (5 minutes)

Import what we need and define the User-Agent: Bluesky asks for no key, but every request still says who sent it, as Wikipedia's did. Every request today goes to Bluesky's public **AppView**, `https://public.api.bsky.app/xrpc/`, the service that answers reads for anyone ([API hosts](https://docs.bsky.app/docs/advanced-guides/api-directory)). Each method's name is part of the address: `app.bsky.actor.getProfile` reads a profile, `app.bsky.feed.getAuthorFeed` an account's posts.

In [ ]:
# Import requests, pandas (as pd), and matplotlib.pyplot (as plt)
# Define HEADERS, whose "User-Agent" is "WebDataScience/1.0 (your-email@colorado.edu)", with your email
# bsky_base = "https://public.api.bsky.app/xrpc"
# Your code here

<details>
<summary>💬 Discuss</summary>

- Census and FRED wanted a key; Bluesky doesn't. What does a key let an agency do that Bluesky gives up?
- Bluesky's protocol lets anyone run an AppView of their own. What would it mean for a researcher if the company's went away?

</details>

## Step 1 — From the website to the API: one agency's profile (8 minutes)

Open the National Weather Service's profile on the website, [bsky.app/profile/nws.noaa.gov](https://bsky.app/profile/nws.noaa.gov). Its **handle**, `nws.noaa.gov`, is a domain name: an account can take a domain as its handle only by proving it controls that domain ([Bluesky's guide](https://bsky.social/about/blog/4-28-2023-domain-handle-tutorial)), so no one but NOAA can post as `nws.noaa.gov`. The blue check beside its name is a second proof: Bluesky itself has **verified** the account ([how verification works](https://bsky.social/about/blog/04-21-2025-verification)). Now ask the API for the same page with [`app.bsky.actor.getProfile`](https://docs.bsky.app/docs/api/app-bsky-actor-get-profile), and find the website's numbers in its answer, as the chapter's ["Bluesky via AT Protocol"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-12-social.html#bluesky-via-at-protocol) reads a profile.

In [ ]:
# profile: the JSON of f"{bsky_base}/app.bsky.actor.getProfile" with params={"actor": "nws.noaa.gov"} and HEADERS
# Print its handle, displayName, followersCount, postsCount, and createdAt; then its "verification"
# Your code here

<details>
<summary>💬 Discuss</summary>

- Did the API's counts match the website's? If they differ by a few, why might that be?
- A search for "National Park Service" on Bluesky finds "Alt National Park Service" and "The National Park Service UNCENSORED", with no `.gov` handle and no check. What would a researcher who collected their posts as the agency's have got wrong?
- Which proof would you trust more, the domain or the check, and why?

</details>

## Step 2 — Five agencies in one table (8 minutes)

Five federal accounts have a `.gov` handle on Bluesky as of November 2026; four of them are NOAA's. Ask for each profile in a loop, keep the same five fields, and make a DataFrame, sorted by followers, as week 11 compared places. [`app.bsky.actor.getProfiles`](https://docs.bsky.app/docs/api/app-bsky-actor-get-profiles) can ask for up to 25 at once; one at a time keeps the loop plain.

In [ ]:
# agencies: "noaa.gov", "nws.noaa.gov", "climate.noaa.gov", "noaacomms.noaa.gov", "federalreserve.gov"
# rows: for each handle, request its profile, and append a dict of handle, displayName, followersCount, postsCount, createdAt[:10]
# accounts: a DataFrame of rows, sorted by followersCount, largest first; show it
# Your code here

### ⏱ Microsprint (3 minutes, alone or in pairs)

Not every official account has a domain handle. The Library of Congress posts as `librarycongress.bsky.social`, and the Federal Reserve Bank of St. Louis, FRED's home in week 11, as `stlouisfed.bsky.social`.

**Do:** add both to the loop, and print each account's `verification` status.
**Share:** which of the seven are verified, and how you'd tell an official `.bsky.social` account from an impostor's.

In [ ]:
# Microsprint: for "librarycongress.bsky.social" and "stlouisfed.bsky.social", request the profile
# Print each handle, its followersCount, and its verification's verifiedStatus
# Your code here

<details>
<summary>💬 Discuss</summary>

- Most federal agencies, the Census Bureau and NASA among them, have no `.gov` handle on Bluesky. What does that say about which agencies a study of "government on Bluesky" would see?
- The National Weather Service has more followers than NOAA, its parent. What might explain it?
- The Federal Reserve joined first but has the fewest followers. Is followers the right measure of an agency's reach?

</details>

## Step 3 — What an agency posted this week (10 minutes)

[`app.bsky.feed.getAuthorFeed`](https://docs.bsky.app/docs/api/app-bsky-feed-get-author-feed) returns an account's posts, newest first, up to 100 a request. With `filter="posts_no_replies"` it leaves out the account's replies, but it still includes its **reposts**: other people's posts it shared, which carry a `"reason"`. Those authors may be private people, so skip any item with a reason, and keep each post's time, its counts, and the start of its text.

In [ ]:
# feed: the JSON's "feed" of f"{bsky_base}/app.bsky.feed.getAuthorFeed", params actor="nws.noaa.gov", limit=25, filter="posts_no_replies"
# posts: for each item without a "reason" (not a repost), a dict of its post's indexedAt[:16], likeCount, repostCount, replyCount, and record text[:70]
# nws_posts: a DataFrame of posts; show it sorted by repostCount, largest first, the top 5
# Your code here

### ⏱ Microsprint (4 minutes, alone or in pairs)

**Do:** run the same request for another of the five agencies.
**Share:** the agency, how many of its 25 items were its own posts rather than reposts, and the topic of its most-reposted post.

In [ ]:
# Microsprint: set params["actor"] to another agency; request again, build the DataFrame the same way, and show its top 3 by reposts
# Your code here

<details>
<summary>💬 Discuss</summary>

- Which kinds of posts got reposted most: warnings, explainers, or announcements? What would you need to say a kind of post *causes* reposts?
- We skipped reposts to keep private people's posts out of our table. What does an agency's reposting tell you that its own posts don't, and how could you study it without printing anyone's handle?
- The feed's last item came with a `cursor`. What do you think it's for? (Week 10's `continue` is a hint.)

</details>

## End of Part 1

On Wednesday, Part 2 follows the `cursor` to read more than one request returns, and asks what an agency's followers and posting rhythm can show. It starts with its own setup step, so you don't need today's cells to have run.

**Before Friday:** get your YouTube Data API key, from a personal Google account, and store it as `YOUTUBE_KEY`.